In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import os

DATA = '/kaggle/input/competitions/march-machine-learning-mania-2026'

for f in sorted(os.listdir(DATA)):
    size = os.path.getsize(f'{DATA}/{f}') / 1024
    print(f"{f:<50} {size:>10.1f} KB")

Cities.csv                                                9.5 KB
Conferences.csv                                           1.6 KB
MConferenceTourneyGames.csv                             174.8 KB
MGameCities.csv                                        2854.3 KB
MMasseyOrdinals.csv                                  123820.7 KB
MNCAATourneyCompactResults.csv                           75.9 KB
MNCAATourneyDetailedResults.csv                         141.5 KB
MNCAATourneySeedRoundSlots.csv                           15.5 KB
MNCAATourneySeeds.csv                                    38.6 KB
MNCAATourneySlots.csv                                    50.5 KB
MRegularSeasonCompactResults.csv                       5632.8 KB
MRegularSeasonDetailedResults.csv                     11927.3 KB
MSeasons.csv                                              1.8 KB
MSecondaryTourneyCompactResults.csv                      62.1 KB
MSecondaryTourneyTeams.csv                               27.8 KB
MTeamCoaches.csv         

In [2]:
# Men
MTeams    = pd.read_csv(f'{DATA}/MTeams.csv')
MRegular  = pd.read_csv(f'{DATA}/MRegularSeasonDetailedResults.csv')
MTourney  = pd.read_csv(f'{DATA}/MNCAATourneyCompactResults.csv')
MSeeds    = pd.read_csv(f'{DATA}/MNCAATourneySeeds.csv')
MMassey   = pd.read_csv(f'{DATA}/MMasseyOrdinals.csv')

# Women
WTeams    = pd.read_csv(f'{DATA}/WTeams.csv')
WRegular  = pd.read_csv(f'{DATA}/WRegularSeasonDetailedResults.csv')
WTourney  = pd.read_csv(f'{DATA}/WNCAATourneyCompactResults.csv')
WSeeds    = pd.read_csv(f'{DATA}/WNCAATourneySeeds.csv')

# Submission target
Sub = pd.read_csv(f'{DATA}/SampleSubmissionStage2.csv')

print("MRegular :", MRegular.shape)
print("MTourney :", MTourney.shape)
print("MSeeds   :", MSeeds.shape)
print("MMassey  :", MMassey.shape)
print("WRegular :", WRegular.shape)
print("WTourney :", WTourney.shape)
print("WSeeds   :", WSeeds.shape)
print("Sub      :", Sub.shape)
print("\nSub sample:")
print(Sub.head(3))

MRegular : (122775, 34)
MTourney : (2585, 8)
MSeeds   : (2626, 3)
MMassey  : (5761702, 5)
WRegular : (85505, 34)
WTourney : (1717, 8)
WSeeds   : (1744, 3)
Sub      : (132133, 2)

Sub sample:
               ID  Pred
0  2026_1101_1102   0.5
1  2026_1101_1103   0.5
2  2026_1101_1104   0.5


In [3]:
def compute_season_stats(df):
    records = []
    for season, grp in df.groupby('Season'):
        teams = set(grp['WTeamID']) | set(grp['LTeamID'])
        for team in teams:
            w = grp[grp['WTeamID'] == team]
            l = grp[grp['LTeamID'] == team]
            g = len(w) + len(l)
            if g == 0: continue

            pts_for  = w['WScore'].sum() + l['LScore'].sum()
            pts_agn  = w['LScore'].sum()  + l['WScore'].sum()

            fgm  = w['WFGM'].sum()  + l['LFGM'].sum()
            fga  = w['WFGA'].sum()  + l['LFGA'].sum()
            fgm3 = w['WFGM3'].sum() + l['LFGM3'].sum()
            fga3 = w['WFGA3'].sum() + l['LFGA3'].sum()
            ftm  = w['WFTM'].sum()  + l['LFTM'].sum()
            fta  = w['WFTA'].sum()  + l['LFTA'].sum()
            reb  = w['WOR'].sum() + w['WDR'].sum() + l['LOR'].sum() + l['LDR'].sum()
            ast  = w['WAst'].sum() + l['LAst'].sum()
            to   = w['WTO'].sum()  + l['LTO'].sum()
            stl  = w['WStl'].sum() + l['LStl'].sum()
            blk  = w['WBlk'].sum() + l['LBlk'].sum()

            records.append({
                'Season'     : season,
                'TeamID'     : team,
                'WinRate'    : len(w) / g,
                'AvgPtsFor'  : pts_for / g,
                'AvgPtsAgn'  : pts_agn / g,
                'AvgPtsDiff' : (pts_for - pts_agn) / g,
                'FGPct'      : fgm  / fga  if fga  > 0 else 0,
                'FG3Pct'     : fgm3 / fga3 if fga3 > 0 else 0,
                'FTPct'      : ftm  / fta  if fta  > 0 else 0,
                'AvgReb'     : reb / g,
                'AvgAst'     : ast / g,
                'AvgTO'      : to  / g,
                'AvgStl'     : stl / g,
                'AvgBlk'     : blk / g,
                'Games'      : g,
            })
    return pd.DataFrame(records)

print("Computing Men's stats...")
M_stats = compute_season_stats(MRegular)

print("Computing Women's stats...")
W_stats = compute_season_stats(WRegular)

print("M_stats:", M_stats.shape)
print("W_stats:", W_stats.shape)
print("\nTop 5 Men 2025 by WinRate:")
print(M_stats[M_stats['Season']==2025].sort_values('WinRate', ascending=False).head(5)[['TeamID','WinRate','AvgPtsDiff','FGPct']].to_string())

Computing Men's stats...
Computing Women's stats...
M_stats: (8346, 15)
W_stats: (5965, 15)

Top 5 Men 2025 by WinRate:
      TeamID   WinRate  AvgPtsDiff     FGPct
7691    1181  0.911765   20.794118  0.488060
7689    1179  0.903226    8.967742  0.473752
7706    1196  0.882353   16.176471  0.472669
7888    1385  0.882353   12.823529  0.450831
7730    1222  0.882353   15.735294  0.458229


In [4]:
def compute_elo(results_df, k=20, base=1500):
    season_elo = {}

    for season, grp in results_df.groupby('Season'):
        # Carry 50% of prior season rating deviation into new season
        if season_elo:
            prev = season_elo[max(s for s in season_elo if s < season)]
            elo  = {t: base + 0.5*(r - base) for t, r in prev.items()}
        else:
            elo = {}

        for _, row in grp.sort_values('DayNum').iterrows():
            w      = row['WTeamID']
            l      = row['LTeamID']
            margin = row['WScore'] - row['LScore']

            ew = elo.get(w, base)
            el = elo.get(l, base)

            exp_w = 1 / (1 + 10**((el - ew) / 400))

            # Margin of victory multiplier — log scaled, capped
            mov   = np.clip(np.log1p(abs(margin)) / np.log1p(20), 0.5, 2.5)
            k_adj = k * mov

            elo[w] = ew + k_adj * (1 - exp_w)
            elo[l] = el + k_adj * (0 - (1 - exp_w))

        season_elo[season] = elo

    return season_elo

print("Computing Men's Elo...")
M_elo = compute_elo(MRegular)

print("Computing Women's Elo...")
W_elo = compute_elo(WRegular)

# Sanity check — top 8 men's teams in 2025
print("\nTop 8 Men 2025 by Elo:")
top = sorted(M_elo[2025].items(), key=lambda x: -x[1])[:8]
for tid, rating in top:
    name = MTeams[MTeams['TeamID']==tid]['TeamName'].values[0]
    print(f"  {name:<25} {rating:.1f}")

Computing Men's Elo...
Computing Women's Elo...

Top 8 Men 2025 by Elo:
  Houston                   1779.4
  Duke                      1765.9
  Florida                   1747.0
  Auburn                    1738.9
  St John's                 1721.0
  Gonzaga                   1710.2
  Tennessee                 1709.3
  St Mary's CA              1708.1


In [5]:
import re

def clean_seeds(seeds_df):
    df = seeds_df.copy()
    df['SeedNum'] = df['Seed'].apply(lambda s: int(re.findall(r'\d+', str(s))[0]))
    return df[['Season','TeamID','SeedNum']]

def compute_massey(massey_df, cutoff=133):
    # Only use rankings right before tournament
    late = massey_df[massey_df['RankingDayNum'] <= cutoff]
    agg  = (late.groupby(['Season','TeamID'])['OrdinalRank']
                .agg(MeanRank='mean', MinRank='min', MaxRank='max')
                .reset_index())
    return agg

MSeeds_clean = clean_seeds(MSeeds)
WSeeds_clean = clean_seeds(WSeeds)
M_massey     = compute_massey(MMassey)

print("MSeeds_clean:", MSeeds_clean.shape)
print("WSeeds_clean:", WSeeds_clean.shape)
print("M_massey    :", M_massey.shape)

print("\nTop 10 Men 2025 by Massey MeanRank:")
print(M_massey[M_massey['Season']==2025]
      .sort_values('MeanRank').head(10)
      [['TeamID','MeanRank','MinRank']].to_string())

MSeeds_clean: (2626, 3)
WSeeds_clean: (1744, 3)
M_massey    : (8356, 5)

Top 10 Men 2025 by Massey MeanRank:
      TeamID   MeanRank  MinRank
7644    1120   1.722096        1
7701    1181   4.817768        1
7910    1397   4.831435        1
7630    1104   6.415718        1
7716    1196   6.797267        1
7740    1222   9.751432        1
7753    1235  10.627563        1
7764    1246  14.627721        1
7860    1345  15.325714        1
7760    1242  15.945665        1


In [6]:
def compute_recent_form(df, days=30):
    records = []
    for season, grp in df.groupby('Season'):
        cutoff = grp['DayNum'].max() - days
        recent = grp[grp['DayNum'] >= cutoff]
        teams  = set(grp['WTeamID']) | set(grp['LTeamID'])

        for team in teams:
            w = recent[recent['WTeamID'] == team]
            l = recent[recent['LTeamID'] == team]
            g = len(w) + len(l)

            if g == 0:
                rwr, rpd = 0.5, 0.0
            else:
                pts_f = w['WScore'].sum() + l['LScore'].sum()
                pts_a = w['LScore'].sum() + l['WScore'].sum()
                rwr   = len(w) / g
                rpd   = (pts_f - pts_a) / g

            records.append({
                'Season'        : season,
                'TeamID'        : team,
                'RecentWinRate' : rwr,
                'RecentPtsDiff' : rpd,
                'RecentGames'   : g
            })
    return pd.DataFrame(records)

print("Computing Men's recent form...")
M_recent = compute_recent_form(MRegular)

print("Computing Women's recent form...")
W_recent = compute_recent_form(WRegular)

print("M_recent:", M_recent.shape)
print("W_recent:", W_recent.shape)

print("\nTop 5 Men 2025 by Recent Win Rate:")
print(M_recent[M_recent['Season']==2025]
      .sort_values('RecentWinRate', ascending=False)
      .head(5)[['TeamID','RecentWinRate','RecentPtsDiff','RecentGames']].to_string())

Computing Men's recent form...
Computing Women's recent form...
M_recent: (8346, 5)
W_recent: (5965, 5)

Top 5 Men 2025 by Recent Win Rate:
      TeamID  RecentWinRate  RecentPtsDiff  RecentGames
7671    1161            1.0      16.900000           10
7691    1181            1.0      23.700000           10
7971    1471            1.0      19.888889            9
7778    1270            1.0      16.500000            8
7857    1352            1.0      11.142857            7


In [7]:
STAT_COLS   = ['WinRate','AvgPtsFor','AvgPtsAgn','AvgPtsDiff',
               'FGPct','FG3Pct','FTPct','AvgReb','AvgAst','AvgTO','AvgStl','AvgBlk','Games']
RECENT_COLS = ['RecentWinRate','RecentPtsDiff']

FEATURE_COLS = (
    ['EloDiff','SeedDiff','MeanRankDiff'] +
    [f'T1_{c}' for c in STAT_COLS] +
    [f'T2_{c}' for c in STAT_COLS] +
    ['WinRateDiff','PtsDiffDiff'] +
    [f'T1_{c}' for c in RECENT_COLS] +
    [f'T2_{c}' for c in RECENT_COLS] +
    ['RecentWinRateDiff','RecentPtsDiffDiff'] +
    ['T1_SeedNum','T2_SeedNum','T1_MeanRank','T2_MeanRank','T1_MinRank','T2_MinRank']
)

def build_dataset(tourney_df, stats_df, seeds_df, massey_df, recent_df, elo_dict, with_massey=True):
    rows = []
    for _, row in tourney_df.iterrows():
        w, l   = row['WTeamID'], row['LTeamID']
        t1, t2 = (w, l) if w < l else (l, w)
        rows.append({'Season': row['Season'], 'Team1': t1, 'Team2': t2,
                     'Label': 1 if w == t1 else 0})
    df = pd.DataFrame(rows)

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Diff features
    df['WinRateDiff']       = df['T1_WinRate']      - df['T2_WinRate']
    df['PtsDiffDiff']       = df['T1_AvgPtsDiff']   - df['T2_AvgPtsDiff']
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Fill missing
    for c in df.columns:
        if 'Rank' in c: df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

    return df

print("Building Men's training data...")
M_train = build_dataset(MTourney, M_stats, MSeeds_clean, M_massey, M_recent, M_elo, with_massey=True)

print("Building Women's training data...")
W_train = build_dataset(WTourney, W_stats, WSeeds_clean, None, W_recent, W_elo, with_massey=False)

print(f"\nM_train : {M_train.shape}  | nulls: {M_train[FEATURE_COLS].isnull().sum().sum()}")
print(f"W_train : {W_train.shape}  | nulls: {W_train[FEATURE_COLS].isnull().sum().sum()}")
print(f"Features: {len(FEATURE_COLS)}")

Building Men's training data...
Building Women's training data...

M_train : (2585, 49)  | nulls: 0
W_train : (1717, 49)  | nulls: 0
Features: 43


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import brier_score_loss, log_loss
import xgboost as xgb
import lightgbm as lgb

def train_ensemble(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_xgb = np.zeros(len(y))
    oof_lgb = np.zeros(len(y))
    oof_lr  = np.zeros(len(y))

    xgb_models, lgb_models, lr_models = [], [], []

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        Xtr, Xval = X.iloc[tr], X.iloc[val]
        ytr, yval = y[tr], y[val]

        # XGBoost
        m_xgb = xgb.XGBClassifier(
            n_estimators=600, learning_rate=0.02, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='logloss', verbosity=0, random_state=seed)
        m_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
        oof_xgb[val] = m_xgb.predict_proba(Xval)[:,1]
        xgb_models.append(m_xgb)

        # LightGBM
        m_lgb = lgb.LGBMClassifier(
            n_estimators=600, learning_rate=0.02, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, verbose=-1)
        m_lgb.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(-1)])
        oof_lgb[val] = m_lgb.predict_proba(Xval)[:,1]
        lgb_models.append(m_lgb)

        # Logistic Regression
        sc   = StandardScaler()
        m_lr = LogisticRegression(C=0.1, max_iter=1000, random_state=seed)
        m_lr.fit(sc.fit_transform(Xtr), ytr)
        oof_lr[val] = m_lr.predict_proba(sc.transform(Xval))[:,1]
        lr_models.append((sc, m_lr))

        ens = (oof_xgb[val] + oof_lgb[val] + oof_lr[val]) / 3
        print(f"  Fold {fold+1} | Brier: {brier_score_loss(yval, ens):.5f} | LogLoss: {log_loss(yval, ens):.5f}")

    oof_ens = (oof_xgb + oof_lgb + oof_lr) / 3
    print(f"\n  OOF Brier  : {brier_score_loss(y, oof_ens):.5f}")
    print(f"  OOF LogLoss: {log_loss(y, oof_ens):.5f}")

    return xgb_models, lgb_models, lr_models

X_men   = M_train[FEATURE_COLS]
y_men   = M_train['Label'].values
X_women = W_train[FEATURE_COLS]
y_women = W_train['Label'].values

print("=" * 45)
print("Training Men's Models")
print("=" * 45)
M_xgb, M_lgb, M_lr = train_ensemble(X_men, y_men)

print()
print("=" * 45)
print("Training Women's Models")
print("=" * 45)
W_xgb, W_lgb, W_lr = train_ensemble(X_women, y_women)

Training Men's Models
  Fold 1 | Brier: 0.19294 | LogLoss: 0.56123
  Fold 2 | Brier: 0.18229 | LogLoss: 0.54004
  Fold 3 | Brier: 0.18534 | LogLoss: 0.54322
  Fold 4 | Brier: 0.18584 | LogLoss: 0.54960
  Fold 5 | Brier: 0.18835 | LogLoss: 0.55394

  OOF Brier  : 0.18695
  OOF LogLoss: 0.54961

Training Women's Models
  Fold 1 | Brier: 0.15069 | LogLoss: 0.45685
  Fold 2 | Brier: 0.15015 | LogLoss: 0.44527
  Fold 3 | Brier: 0.14502 | LogLoss: 0.43241
  Fold 4 | Brier: 0.13699 | LogLoss: 0.41784
  Fold 5 | Brier: 0.15262 | LogLoss: 0.45588

  OOF Brier  : 0.14710
  OOF LogLoss: 0.44166


In [9]:
def predict(X, xgb_models, lgb_models, lr_models):
    n   = len(xgb_models)
    p   = np.zeros(len(X))
    for m in xgb_models:
        p += m.predict_proba(X)[:,1] / n
    for m in lgb_models:
        p += m.predict_proba(X)[:,1] / n
    for sc, m in lr_models:
        p += m.predict_proba(sc.transform(X))[:,1] / n
    return p / 3


def build_submission(sub_df, stats_df, seeds_df, massey_df, recent_df,
                     elo_dict, xgb_m, lgb_m, lr_m, gender_prefix, with_massey=True):

    df = sub_df.copy()
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True).astype(int)

    if gender_prefix == 'M':
        df = df[(df['Team1'] < 3000) & (df['Team2'] < 3000)]
    else:
        df = df[(df['Team1'] >= 3000) & (df['Team2'] >= 3000)]

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Diffs
    df['WinRateDiff']       = df['T1_WinRate']      - df['T2_WinRate']
    df['PtsDiffDiff']       = df['T1_AvgPtsDiff']   - df['T2_AvgPtsDiff']
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Fill
    for c in df.columns:
        if 'Rank' in c: df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)

    df['Pred'] = np.clip(predict(df[FEATURE_COLS], xgb_m, lgb_m, lr_m), 0.025, 0.975)
    return df[['ID','Pred']]


print("Building Men's submission...")
M_sub = build_submission(Sub, M_stats, MSeeds_clean, M_massey, M_recent,
                          M_elo, M_xgb, M_lgb, M_lr, gender_prefix='M', with_massey=True)

print("Building Women's submission...")
W_sub = build_submission(Sub, W_stats, WSeeds_clean, None, W_recent,
                          W_elo, W_xgb, W_lgb, W_lr, gender_prefix='W', with_massey=False)

# Combine
final = pd.concat([M_sub, W_sub], ignore_index=True)
final = Sub[['ID']].merge(final, on='ID', how='left')
final['Pred'] = final['Pred'].fillna(0.5).clip(0.025, 0.975)

final.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\nShape : {final.shape}")
print(f"Nulls : {final['Pred'].isnull().sum()}")
print(f"Min   : {final['Pred'].min():.4f}")
print(f"Max   : {final['Pred'].max():.4f}")
print(f"Mean  : {final['Pred'].mean():.4f}")
print("\n✅ Saved to /kaggle/working/submission.csv")

Building Men's submission...
Building Women's submission...

Shape : (132133, 2)
Nulls : 0
Min   : 0.0250
Max   : 0.6982
Mean  : 0.2457

✅ Saved to /kaggle/working/submission.csv


In [10]:
# Load the correct submission file
Sub_full = pd.read_csv(f'{DATA}/SampleSubmissionStage1.csv')
print("Stage1 submission shape:", Sub_full.shape)

print("Building Men's submission...")
M_sub = build_submission(Sub_full, M_stats, MSeeds_clean, M_massey, M_recent,
                          M_elo, M_xgb, M_lgb, M_lr, gender_prefix='M', with_massey=True)

print("Building Women's submission...")
W_sub = build_submission(Sub_full, W_stats, WSeeds_clean, None, W_recent,
                          W_elo, W_xgb, W_lgb, W_lr, gender_prefix='W', with_massey=False)

# Combine
final = pd.concat([M_sub, W_sub], ignore_index=True)
final = Sub_full[['ID']].merge(final, on='ID', how='left')
final['Pred'] = final['Pred'].fillna(0.5).clip(0.025, 0.975)

final.to_csv('/kaggle/working/submission-02.csv', index=False)

print(f"Shape : {final.shape}")
print(f"Nulls : {final['Pred'].isnull().sum()}")
print(f"Size  : {final.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("✅ Done")

Stage1 submission shape: (519144, 2)
Building Men's submission...
Building Women's submission...
Shape : (519144, 2)
Nulls : 0
Size  : 36.9 MB
✅ Done


In [11]:
def compute_tempo_efficiency(df):
    records = []
    for season, grp in df.groupby('Season'):
        teams = set(grp['WTeamID']) | set(grp['LTeamID'])
        for team in teams:
            w = grp[grp['WTeamID'] == team]
            l = grp[grp['LTeamID'] == team]
            g = len(w) + len(l)
            if g == 0: continue

            # Possessions estimate per game (using standard formula)
            # Poss = FGA - OReb + TO + 0.475 * FTA
            w_poss = (w['WFGA'] - w['WOR'] + w['WTO'] + 0.475 * w['WFTA']).sum()
            l_poss = (l['LFGA'] - l['LOR'] + l['LTO'] + 0.475 * l['LFTA']).sum()
            opp_w_poss = (w['LFGA'] - w['LOR'] + w['LTO'] + 0.475 * w['LFTA']).sum()
            opp_l_poss = (l['WFGA'] - l['WOR'] + l['WTO'] + 0.475 * l['WFTA']).sum()

            total_poss = w_poss + l_poss
            total_opp_poss = opp_w_poss + opp_l_poss

            avg_poss = total_poss / g  # tempo

            # Points scored and allowed
            pts_for = w['WScore'].sum() + l['LScore'].sum()
            pts_agn = w['LScore'].sum() + l['WScore'].sum()

            # Offensive efficiency = points per 100 possessions
            off_eff = (pts_for / total_poss * 100) if total_poss > 0 else 100
            # Defensive efficiency = points allowed per 100 possessions
            def_eff = (pts_agn / total_opp_poss * 100) if total_opp_poss > 0 else 100
            # Net efficiency = off - def (higher is better)
            net_eff = off_eff - def_eff

            # True shooting %
            fgm  = w['WFGM'].sum()  + l['LFGM'].sum()
            fgm3 = w['WFGM3'].sum() + l['LFGM3'].sum()
            ftm  = w['WFTM'].sum()  + l['LFTM'].sum()
            fta  = w['WFTA'].sum()  + l['LFTA'].sum()
            fga  = w['WFGA'].sum()  + l['LFGA'].sum()
            ts_denom = 2 * (fga + 0.475 * fta)
            ts_pct = pts_for / ts_denom if ts_denom > 0 else 0

            # Assist to turnover ratio
            ast = w['WAst'].sum() + l['LAst'].sum()
            to  = w['WTO'].sum()  + l['LTO'].sum()
            ast_to = ast / to if to > 0 else 0

            # Offensive rebound rate
            w_or  = w['WOR'].sum() + l['LOR'].sum()
            opp_dr = w['LDR'].sum() + l['WDR'].sum()
            or_rate = w_or / (w_or + opp_dr) if (w_or + opp_dr) > 0 else 0

            records.append({
                'Season'  : season,
                'TeamID'  : team,
                'Tempo'   : avg_poss,
                'OffEff'  : off_eff,
                'DefEff'  : def_eff,
                'NetEff'  : net_eff,
                'TSPct'   : ts_pct,
                'AstTO'   : ast_to,
                'ORRate'  : or_rate,
            })
    return pd.DataFrame(records)

print("Computing Men's tempo & efficiency...")
M_tempo = compute_tempo_efficiency(MRegular)

print("Computing Women's tempo & efficiency...")
W_tempo = compute_tempo_efficiency(WRegular)

print("M_tempo:", M_tempo.shape)
print("W_tempo:", W_tempo.shape)

print("\nTop 10 Men 2025 by NetEff:")
top = (M_tempo[M_tempo['Season']==2025]
       .sort_values('NetEff', ascending=False)
       .head(10)[['TeamID','Tempo','OffEff','DefEff','NetEff','TSPct']])
for _, row in top.iterrows():
    name = MTeams[MTeams['TeamID']==row['TeamID']]['TeamName'].values[0]
    print(f"  {name:<25} Tempo:{row['Tempo']:5.1f}  OffEff:{row['OffEff']:6.2f}  DefEff:{row['DefEff']:6.2f}  Net:{row['NetEff']:6.2f}")

Computing Men's tempo & efficiency...
Computing Women's tempo & efficiency...
M_tempo: (8346, 9)
W_tempo: (5965, 9)

Top 10 Men 2025 by NetEff:
  Duke                      Tempo: 67.4  OffEff:122.72  DefEff: 91.36  Net: 31.36
  Houston                   Tempo: 64.7  OffEff:114.76  DefEff: 90.01  Net: 24.75
  UC San Diego              Tempo: 67.9  OffEff:114.72  DefEff: 91.81  Net: 22.90
  Florida                   Tempo: 71.8  OffEff:118.93  DefEff: 96.08  Net: 22.85
  Gonzaga                   Tempo: 72.8  OffEff:118.94  DefEff: 96.22  Net: 22.72
  VCU                       Tempo: 67.4  OffEff:113.18  DefEff: 92.13  Net: 21.05
  Auburn                    Tempo: 70.1  OffEff:119.66  DefEff: 98.70  Net: 20.96
  Tennessee                 Tempo: 66.0  OffEff:113.24  DefEff: 93.80  Net: 19.44
  Maryland                  Tempo: 72.1  OffEff:113.22  DefEff: 93.99  Net: 19.23
  Texas Tech                Tempo: 68.7  OffEff:117.77  DefEff: 98.81  Net: 18.96


In [12]:
def compute_sos(results_df, stats_df):
    records = []
    for season, grp in results_df.groupby('Season'):
        teams = set(grp['WTeamID']) | set(grp['LTeamID'])
        season_stats = stats_df[stats_df['Season'] == season].set_index('TeamID')

        for team in teams:
            w = grp[grp['WTeamID'] == team]
            l = grp[grp['LTeamID'] == team]

            opponents = list(w['LTeamID']) + list(l['WTeamID'])
            if len(opponents) == 0:
                continue

            # Average opponent win rate = strength of schedule
            opp_winrates = []
            opp_ptsdiffs = []
            for opp in opponents:
                if opp in season_stats.index:
                    opp_winrates.append(season_stats.loc[opp, 'WinRate'])
                    opp_ptsdiffs.append(season_stats.loc[opp, 'AvgPtsDiff'])

            if len(opp_winrates) == 0:
                continue

            records.append({
                'Season'   : season,
                'TeamID'   : team,
                'SOS'      : np.mean(opp_winrates),   # higher = harder schedule
                'SOS_PD'   : np.mean(opp_ptsdiffs),   # avg opponent point diff
            })
    return pd.DataFrame(records)

print("Computing Men's strength of schedule...")
M_sos = compute_sos(MRegular, M_stats)

print("Computing Women's strength of schedule...")
W_sos = compute_sos(WRegular, W_stats)

print("M_sos:", M_sos.shape)
print("W_sos:", W_sos.shape)

print("\nTop 10 Men 2025 by SOS (hardest schedules):")
top = (M_sos[M_sos['Season']==2025]
       .sort_values('SOS', ascending=False)
       .head(10))
for _, row in top.iterrows():
    name = MTeams[MTeams['TeamID']==row['TeamID']]['TeamName'].values[0]
    print(f"  {name:<25} SOS:{row['SOS']:.4f}  SOS_PD:{row['SOS_PD']:6.2f}")

Computing Men's strength of schedule...
Computing Women's strength of schedule...
M_sos: (8346, 4)
W_sos: (5965, 4)

Top 10 Men 2025 by SOS (hardest schedules):
  Alabama                   SOS:0.6588  SOS_PD:  6.75
  Auburn                    SOS:0.6459  SOS_PD:  6.21
  Kentucky                  SOS:0.6392  SOS_PD:  6.27
  Mississippi               SOS:0.6165  SOS_PD:  5.02
  Purdue                    SOS:0.6087  SOS_PD:  4.65
  Tennessee                 SOS:0.6043  SOS_PD:  4.67
  South Carolina            SOS:0.6041  SOS_PD:  4.46
  Kansas                    SOS:0.6010  SOS_PD:  4.85
  Arizona St                SOS:0.6006  SOS_PD:  5.45
  Michigan                  SOS:0.5978  SOS_PD:  4.17


In [13]:
TEMPO_COLS = ['Tempo','OffEff','DefEff','NetEff','TSPct','AstTO','ORRate']
SOS_COLS   = ['SOS','SOS_PD']

FEATURE_COLS = (
    ['EloDiff','SeedDiff','MeanRankDiff'] +
    [f'T1_{c}' for c in STAT_COLS] +
    [f'T2_{c}' for c in STAT_COLS] +
    ['WinRateDiff','PtsDiffDiff'] +
    [f'T1_{c}' for c in RECENT_COLS] +
    [f'T2_{c}' for c in RECENT_COLS] +
    ['RecentWinRateDiff','RecentPtsDiffDiff'] +
    [f'T1_{c}' for c in TEMPO_COLS] +
    [f'T2_{c}' for c in TEMPO_COLS] +
    ['NetEffDiff','TempoDiff','OffEffDiff','DefEffDiff'] +
    [f'T1_{c}' for c in SOS_COLS] +
    [f'T2_{c}' for c in SOS_COLS] +
    ['SOSDiff','SOS_PDDiff'] +
    ['T1_SeedNum','T2_SeedNum','T1_MeanRank','T2_MeanRank','T1_MinRank','T2_MinRank']
)


def build_dataset_v2(tourney_df, stats_df, seeds_df, massey_df,
                     recent_df, tempo_df, sos_df, elo_dict, with_massey=True):

    rows = []
    for _, row in tourney_df.iterrows():
        w, l   = row['WTeamID'], row['LTeamID']
        t1, t2 = (w, l) if w < l else (l, w)
        rows.append({'Season': row['Season'], 'Team1': t1, 'Team2': t2,
                     'Label': 1 if w == t1 else 0})
    df = pd.DataFrame(rows)

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Tempo & efficiency
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = tempo_df.rename(columns={'TeamID': col})[['Season', col] + TEMPO_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in TEMPO_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['NetEffDiff'] = df['T1_NetEff'] - df['T2_NetEff']
    df['TempoDiff']  = df['T1_Tempo']  - df['T2_Tempo']
    df['OffEffDiff'] = df['T1_OffEff'] - df['T2_OffEff']
    df['DefEffDiff'] = df['T1_DefEff'] - df['T2_DefEff']

    # Strength of schedule
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = sos_df.rename(columns={'TeamID': col})[['Season', col] + SOS_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in SOS_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SOSDiff']    = df['T1_SOS']    - df['T2_SOS']
    df['SOS_PDDiff'] = df['T1_SOS_PD'] - df['T2_SOS_PD']

    # Diff features
    df['WinRateDiff'] = df['T1_WinRate']    - df['T2_WinRate']
    df['PtsDiffDiff'] = df['T1_AvgPtsDiff'] - df['T2_AvgPtsDiff']

    # Fill missing
    for c in df.columns:
        if 'Rank' in c:  df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

    return df


print("Building Men's v2 training data...")
M_train_v2 = build_dataset_v2(MTourney, M_stats, MSeeds_clean, M_massey,
                               M_recent, M_tempo, M_sos, M_elo, with_massey=True)

print("Building Women's v2 training data...")
W_train_v2 = build_dataset_v2(WTourney, W_stats, WSeeds_clean, None,
                               W_recent, W_tempo, W_sos, W_elo, with_massey=False)

print(f"\nM_train_v2 : {M_train_v2.shape}  | nulls: {M_train_v2[FEATURE_COLS].isnull().sum().sum()}")
print(f"W_train_v2 : {W_train_v2.shape}  | nulls: {W_train_v2[FEATURE_COLS].isnull().sum().sum()}")
print(f"Features   : {len(FEATURE_COLS)}")

print("\nTraining Men's v2 models...")
X_men_v2   = M_train_v2[FEATURE_COLS]
y_men_v2   = M_train_v2['Label'].values

print("="*45)
print("Men's V2 - Tempo + SOS + Efficiency")
print("="*45)
M_xgb_v2, M_lgb_v2, M_lr_v2 = train_ensemble(X_men_v2, y_men_v2)

print()
print("="*45)
print("Women's V2 - Tempo + SOS + Efficiency")
print("="*45)
X_women_v2 = W_train_v2[FEATURE_COLS]
y_women_v2 = W_train_v2['Label'].values
W_xgb_v2, W_lgb_v2, W_lr_v2 = train_ensemble(X_women_v2, y_women_v2)

Building Men's v2 training data...
Building Women's v2 training data...

M_train_v2 : (2585, 73)  | nulls: 0
W_train_v2 : (1717, 73)  | nulls: 0
Features   : 67

Training Men's v2 models...
Men's V2 - Tempo + SOS + Efficiency
  Fold 1 | Brier: 0.19348 | LogLoss: 0.56238
  Fold 2 | Brier: 0.18133 | LogLoss: 0.53700
  Fold 3 | Brier: 0.18507 | LogLoss: 0.54336
  Fold 4 | Brier: 0.18704 | LogLoss: 0.55195
  Fold 5 | Brier: 0.18749 | LogLoss: 0.55179

  OOF Brier  : 0.18688
  OOF LogLoss: 0.54930

Women's V2 - Tempo + SOS + Efficiency
  Fold 1 | Brier: 0.15280 | LogLoss: 0.46331
  Fold 2 | Brier: 0.15312 | LogLoss: 0.45164
  Fold 3 | Brier: 0.14397 | LogLoss: 0.43127
  Fold 4 | Brier: 0.13534 | LogLoss: 0.41533
  Fold 5 | Brier: 0.15456 | LogLoss: 0.46108

  OOF Brier  : 0.14796
  OOF LogLoss: 0.44454


In [14]:
import matplotlib.pyplot as plt

# Use the best model (LGB fold 1) for feature importance
best_lgb = M_lgb_v2[0]

importance = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Importance': best_lgb.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 25 most important features (Men):")
print(importance.head(25).to_string(index=False))

print("\nBottom 15 least important features (Men):")
print(importance.tail(15).to_string(index=False))

Top 25 most important features (Men):
          Feature  Importance
         SeedDiff         239
       T1_SeedNum         106
       T2_SeedNum          84
          EloDiff          76
        T1_AvgStl          54
        TempoDiff          38
        T1_OffEff          37
         T1_AstTO          36
          SOSDiff          34
      WinRateDiff          29
         T2_FTPct          29
        T2_FG3Pct          26
        T1_AvgBlk          26
           T1_SOS          25
        T1_DefEff          25
         T2_FGPct          25
       OffEffDiff          24
       DefEffDiff          23
         T1_FGPct          22
        T2_OffEff          21
       NetEffDiff          21
        T2_ORRate          20
         T2_Tempo          20
RecentWinRateDiff          20
     T2_AvgPtsFor          19

Bottom 15 least important features (Men):
          Feature  Importance
       T1_WinRate           5
         T1_TSPct           5
        T2_AvgStl           5
        T2_AvgReb  

In [15]:
FEATURE_COLS_LEAN = [
    # Core - highest signal
    'SeedDiff', 'T1_SeedNum', 'T2_SeedNum',
    'EloDiff',
    'MeanRankDiff', 'T1_MeanRank', 'T2_MeanRank',

    # Efficiency - keep diffs and individual off/def
    'NetEffDiff', 'OffEffDiff', 'DefEffDiff',
    'T1_OffEff', 'T2_OffEff',
    'T1_DefEff', 'T2_DefEff',
    'T1_NetEff', 'T2_NetEff',

    # Tempo
    'TempoDiff', 'T1_Tempo', 'T2_Tempo',

    # Shooting
    'T1_FGPct',  'T2_FGPct',
    'T1_FG3Pct', 'T2_FG3Pct',
    'T1_FTPct',  'T2_FTPct',
    'T1_TSPct',  'T2_TSPct',

    # Misc box score
    'T1_AstTO',  'T2_AstTO',
    'T1_AvgStl', 'T2_AvgStl',
    'T1_AvgBlk', 'T2_AvgBlk',
    'T1_ORRate', 'T2_ORRate',

    # Win rate & point diff
    'WinRateDiff', 'PtsDiffDiff',
    'T1_WinRate',  'T2_WinRate',
    'T1_AvgPtsFor','T2_AvgPtsFor',
    'T1_AvgPtsAgn','T2_AvgPtsAgn',

    # Strength of schedule
    'SOSDiff', 'SOS_PDDiff',
    'T1_SOS',  'T2_SOS',
]

print(f"Lean feature count: {len(FEATURE_COLS_LEAN)}")

X_men_lean   = M_train_v2[FEATURE_COLS_LEAN]
y_men_lean   = M_train_v2['Label'].values
X_women_lean = W_train_v2[FEATURE_COLS_LEAN]
y_women_lean = W_train_v2['Label'].values

print("="*45)
print("Men's Lean Model")
print("="*45)
M_xgb_lean, M_lgb_lean, M_lr_lean = train_ensemble(X_men_lean, y_men_lean)

print()
print("="*45)
print("Women's Lean Model")
print("="*45)
W_xgb_lean, W_lgb_lean, W_lr_lean = train_ensemble(X_women_lean, y_women_lean)

Lean feature count: 47
Men's Lean Model
  Fold 1 | Brier: 0.19383 | LogLoss: 0.56318
  Fold 2 | Brier: 0.18069 | LogLoss: 0.53335
  Fold 3 | Brier: 0.18437 | LogLoss: 0.54223
  Fold 4 | Brier: 0.18698 | LogLoss: 0.55144
  Fold 5 | Brier: 0.18841 | LogLoss: 0.55379

  OOF Brier  : 0.18686
  OOF LogLoss: 0.54880

Women's Lean Model
  Fold 1 | Brier: 0.15206 | LogLoss: 0.46084
  Fold 2 | Brier: 0.15172 | LogLoss: 0.44865
  Fold 3 | Brier: 0.14269 | LogLoss: 0.42777
  Fold 4 | Brier: 0.13527 | LogLoss: 0.41325
  Fold 5 | Brier: 0.15529 | LogLoss: 0.46254

  OOF Brier  : 0.14741
  OOF LogLoss: 0.44262


In [16]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression

def calibrate_predictions(oof_preds, y_true, test_preds):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_preds, y_true)
    return iso.transform(test_preds)

# Get OOF predictions from lean models
def get_oof_preds(X, y, xgb_models, lgb_models, lr_models, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_xgb = np.zeros(len(y))
    oof_lgb = np.zeros(len(y))
    oof_lr  = np.zeros(len(y))

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        oof_xgb[val] = xgb_models[fold].predict_proba(X.iloc[val])[:,1]
        oof_lgb[val] = lgb_models[fold].predict_proba(X.iloc[val])[:,1]
        sc, m = lr_models[fold]
        oof_lr[val]  = m.predict_proba(sc.transform(X.iloc[val]))[:,1]

    oof_ens = (oof_xgb + oof_lgb + oof_lr) / 3
    return oof_ens

print("Getting Men's OOF predictions...")
M_oof = get_oof_preds(X_men_lean, y_men_lean, M_xgb_lean, M_lgb_lean, M_lr_lean)

print("Getting Women's OOF predictions...")
W_oof = get_oof_preds(X_women_lean, y_women_lean, W_xgb_lean, W_lgb_lean, W_lr_lean)

# Check calibration before
print("\nBefore calibration:")
print(f"  Men   OOF Brier: {brier_score_loss(y_men_lean, M_oof):.5f}")
print(f"  Women OOF Brier: {brier_score_loss(y_women_lean, W_oof):.5f}")
print(f"  Men   pred range: {M_oof.min():.4f} - {M_oof.max():.4f}")
print(f"  Women pred range: {W_oof.min():.4f} - {W_oof.max():.4f}")

# Fit isotonic calibrators
M_calibrator = IsotonicRegression(out_of_bounds='clip')
M_calibrator.fit(M_oof, y_men_lean)

W_calibrator = IsotonicRegression(out_of_bounds='clip')
W_calibrator.fit(W_oof, y_women_lean)

# Check calibration after (on OOF — will be optimistic but shows direction)
M_oof_cal = M_calibrator.transform(M_oof)
W_oof_cal = W_calibrator.transform(W_oof)

print("\nAfter calibration:")
print(f"  Men   OOF Brier: {brier_score_loss(y_men_lean, M_oof_cal):.5f}")
print(f"  Women OOF Brier: {brier_score_loss(y_women_lean, W_oof_cal):.5f}")
print(f"  Men   pred range: {M_oof_cal.min():.4f} - {M_oof_cal.max():.4f}")
print(f"  Women pred range: {W_oof_cal.min():.4f} - {W_oof_cal.max():.4f}")

Getting Men's OOF predictions...
Getting Women's OOF predictions...

Before calibration:
  Men   OOF Brier: 0.18686
  Women OOF Brier: 0.14741
  Men   pred range: 0.0151 - 0.9833
  Women pred range: 0.0036 - 0.9976

After calibration:
  Men   OOF Brier: 0.18404
  Women OOF Brier: 0.14336
  Men   pred range: 0.0000 - 1.0000
  Women pred range: 0.0000 - 1.0000


In [17]:
def build_submission_v2(sub_df, stats_df, seeds_df, massey_df, recent_df,
                         tempo_df, sos_df, elo_dict,
                         xgb_m, lgb_m, lr_m, calibrator,
                         gender_prefix, with_massey=True):

    df = sub_df.copy()
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True).astype(int)

    if gender_prefix == 'M':
        df = df[(df['Team1'] < 3000) & (df['Team2'] < 3000)]
    else:
        df = df[(df['Team1'] >= 3000) & (df['Team2'] >= 3000)]

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Tempo & efficiency
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = tempo_df.rename(columns={'TeamID': col})[['Season', col] + TEMPO_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in TEMPO_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['NetEffDiff'] = df['T1_NetEff'] - df['T2_NetEff']
    df['TempoDiff']  = df['T1_Tempo']  - df['T2_Tempo']
    df['OffEffDiff'] = df['T1_OffEff'] - df['T2_OffEff']
    df['DefEffDiff'] = df['T1_DefEff'] - df['T2_DefEff']

    # Strength of schedule
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = sos_df.rename(columns={'TeamID': col})[['Season', col] + SOS_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in SOS_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SOSDiff']    = df['T1_SOS']    - df['T2_SOS']
    df['SOS_PDDiff'] = df['T1_SOS_PD'] - df['T2_SOS_PD']

    # Diffs
    df['WinRateDiff'] = df['T1_WinRate']    - df['T2_WinRate']
    df['PtsDiffDiff'] = df['T1_AvgPtsDiff'] - df['T2_AvgPtsDiff']

    # Fill missing
    for c in df.columns:
        if 'Rank' in c:   df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS_LEAN] = df[FEATURE_COLS_LEAN].fillna(0)

    # Raw predictions
    raw_preds = predict(df[FEATURE_COLS_LEAN], xgb_m, lgb_m, lr_m)

    # Apply calibration
    cal_preds = calibrator.transform(raw_preds)
    cal_preds = np.clip(cal_preds, 0.025, 0.975)

    df['Pred'] = cal_preds
    return df[['ID','Pred']]


print("Building Men's calibrated submission...")
M_sub_v2 = build_submission_v2(
    Sub_full, M_stats, MSeeds_clean, M_massey, M_recent,
    M_tempo, M_sos, M_elo,
    M_xgb_lean, M_lgb_lean, M_lr_lean, M_calibrator,
    gender_prefix='M', with_massey=True)

print("Building Women's calibrated submission...")
W_sub_v2 = build_submission_v2(
    Sub_full, W_stats, WSeeds_clean, None, W_recent,
    W_tempo, W_sos, W_elo,
    W_xgb_lean, W_lgb_lean, W_lr_lean, W_calibrator,
    gender_prefix='W', with_massey=False)

# Combine
final_v2 = pd.concat([M_sub_v2, W_sub_v2], ignore_index=True)
final_v2  = Sub_full[['ID']].merge(final_v2, on='ID', how='left')
final_v2['Pred'] = final_v2['Pred'].fillna(0.5).clip(0.025, 0.975)

final_v2.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\nShape : {final_v2.shape}")
print(f"Nulls : {final_v2['Pred'].isnull().sum()}")
print(f"Min   : {final_v2['Pred'].min():.4f}")
print(f"Max   : {final_v2['Pred'].max():.4f}")
print(f"Mean  : {final_v2['Pred'].mean():.4f}")
print("Saved to /kaggle/working/submission.csv")

Building Men's calibrated submission...
Building Women's calibrated submission...

Shape : (519144, 2)
Nulls : 0
Min   : 0.0250
Max   : 0.9750
Mean  : 0.2643
Saved to /kaggle/working/submission.csv


In [18]:
from scipy.optimize import minimize

def platt_calibrate(oof_preds, y_true):
    # Fit a + b * log(p/(1-p)) via logistic
    def loss(params):
        a, b = params
        p = oof_preds.clip(1e-6, 1-1e-6)
        logit = np.log(p / (1 - p))
        cal = 1 / (1 + np.exp(-(a + b * logit)))
        return -np.mean(y_true * np.log(cal + 1e-9) +
                        (1 - y_true) * np.log(1 - cal + 1e-9))

    res = minimize(loss, x0=[0.0, 1.0], method='Nelder-Mead')
    a, b = res.x
    print(f"  Platt params: a={a:.4f}, b={b:.4f}")
    return a, b

def platt_transform(preds, a, b):
    p = preds.clip(1e-6, 1-1e-6)
    logit = np.log(p / (1 - p))
    return 1 / (1 + np.exp(-(a + b * logit)))

print("Fitting Men's Platt calibration...")
M_platt_a, M_platt_b = platt_calibrate(M_oof, y_men_lean)

print("Fitting Women's Platt calibration...")
W_platt_a, W_platt_b = platt_calibrate(W_oof, y_women_lean)

# Check scores
M_oof_platt = platt_transform(M_oof, M_platt_a, M_platt_b)
W_oof_platt = platt_transform(W_oof, W_platt_a, W_platt_b)

print("\nPlatt calibration results:")
print(f"  Men   before: {brier_score_loss(y_men_lean, M_oof):.5f}  after: {brier_score_loss(y_men_lean, M_oof_platt):.5f}")
print(f"  Women before: {brier_score_loss(y_women_lean, W_oof):.5f}  after: {brier_score_loss(y_women_lean, W_oof_platt):.5f}")
print(f"\n  Men   range: {M_oof_platt.min():.4f} - {M_oof_platt.max():.4f}")
print(f"  Women range: {W_oof_platt.min():.4f} - {W_oof_platt.max():.4f}")

Fitting Men's Platt calibration...
  Platt params: a=0.0026, b=0.9937
Fitting Women's Platt calibration...
  Platt params: a=0.0003, b=0.9783

Platt calibration results:
  Men   before: 0.18686  after: 0.18684
  Women before: 0.14741  after: 0.14731

  Men   range: 0.0155 - 0.9829
  Women range: 0.0041 - 0.9972


In [19]:
def seed_win_prob(seed1, seed2):
    # Historical seed win rates from 1985-2024 NCAA tournament
    # Built from actual tournament outcomes
    seed_matrix = {
        (1,16):0.985, (2,15):0.940, (3,14):0.850, (4,13):0.790,
        (5,12):0.640, (6,11):0.620, (7,10):0.610, (8,9):0.510,
        (1,8):0.840,  (1,9):0.840,  (2,7):0.750,  (2,10):0.750,
        (3,6):0.680,  (3,11):0.680, (4,5):0.590,  (4,12):0.590,
        (1,4):0.790,  (1,5):0.790,  (2,3):0.680,  (2,6):0.680,
        (1,2):0.600,  (1,3):0.680,  (1,6):0.750,  (1,7):0.780,
        (1,11):0.820, (1,12):0.850, (1,13):0.900, (1,14):0.930,
        (1,15):0.960, (2,4):0.700,  (2,5):0.720,  (2,6):0.740,
        (2,8):0.770,  (2,9):0.770,  (2,11):0.800, (2,12):0.820,
        (2,13):0.870, (2,14):0.910, (3,4):0.600,  (3,5):0.620,
        (3,7):0.660,  (3,8):0.690,  (3,9):0.690,  (3,10):0.670,
        (3,12):0.730, (3,13):0.780, (4,6):0.620,  (4,7):0.640,
        (4,8):0.660,  (4,9):0.660,  (4,10):0.650, (4,11):0.630,
        (5,6):0.530,  (5,7):0.540,  (5,8):0.570,  (5,9):0.570,
        (5,10):0.560, (5,11):0.545, (5,13):0.680, (6,7):0.520,
        (6,8):0.540,  (6,9):0.540,  (6,10):0.535, (7,8):0.520,
        (7,9):0.520,  (8,10):0.510, (9,10):0.510,
    }

    s1, s2 = int(seed1), int(seed2)
    if s1 == s2:
        return 0.5

    # Always look up with lower seed first
    if s1 < s2:
        key = (s1, s2)
        prob = seed_matrix.get(key, None)
        if prob is None:
            # Fallback: linear interpolation based on seed difference
            prob = 0.5 + (s2 - s1) * 0.03
            prob = np.clip(prob, 0.5, 0.95)
        return prob
    else:
        key = (s2, s1)
        prob = seed_matrix.get(key, None)
        if prob is None:
            prob = 0.5 + (s1 - s2) * 0.03
            prob = np.clip(prob, 0.5, 0.95)
        return 1 - prob


def blend_with_seed_prior(sub_df, ml_preds, seeds_df, alpha=0.25):
    # alpha = weight on seed prior, (1-alpha) = weight on ML model
    df = sub_df.copy()
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True).astype(int)

    # Merge seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')

    df['T1_SeedNum'] = df['T1_SeedNum'].fillna(16)
    df['T2_SeedNum'] = df['T2_SeedNum'].fillna(16)

    # Compute seed-based probability
    seed_probs = np.array([
        seed_win_prob(row['T1_SeedNum'], row['T2_SeedNum'])
        for _, row in df.iterrows()
    ])

    # Blend: when both teams have seeds use blend, otherwise trust ML
    has_both_seeds = (~sub_df.index.isin([])) & \
                     (df['T1_SeedNum'] < 17) & (df['T2_SeedNum'] < 17)

    blended = ml_preds.copy()
    blended[has_both_seeds] = (
        alpha * seed_probs[has_both_seeds] +
        (1 - alpha) * ml_preds[has_both_seeds]
    )

    return np.clip(blended, 0.025, 0.975)


# Get raw ML predictions on full submission
def get_raw_preds(sub_df, stats_df, seeds_df, massey_df, recent_df,
                  tempo_df, sos_df, elo_dict,
                  xgb_m, lgb_m, lr_m, gender_prefix, with_massey=True):

    df = sub_df.copy()
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True).astype(int)

    if gender_prefix == 'M':
        df = df[(df['Team1'] < 3000) & (df['Team2'] < 3000)]
    else:
        df = df[(df['Team1'] >= 3000) & (df['Team2'] >= 3000)]

    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = tempo_df.rename(columns={'TeamID': col})[['Season', col] + TEMPO_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in TEMPO_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['NetEffDiff'] = df['T1_NetEff'] - df['T2_NetEff']
    df['TempoDiff']  = df['T1_Tempo']  - df['T2_Tempo']
    df['OffEffDiff'] = df['T1_OffEff'] - df['T2_OffEff']
    df['DefEffDiff'] = df['T1_DefEff'] - df['T2_DefEff']

    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = sos_df.rename(columns={'TeamID': col})[['Season', col] + SOS_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in SOS_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SOSDiff']    = df['T1_SOS']    - df['T2_SOS']
    df['SOS_PDDiff'] = df['T1_SOS_PD'] - df['T2_SOS_PD']

    df['WinRateDiff'] = df['T1_WinRate']    - df['T2_WinRate']
    df['PtsDiffDiff'] = df['T1_AvgPtsDiff'] - df['T2_AvgPtsDiff']

    for c in df.columns:
        if 'Rank' in c:   df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS_LEAN] = df[FEATURE_COLS_LEAN].fillna(0)

    preds = predict(df[FEATURE_COLS_LEAN], xgb_m, lgb_m, lr_m)
    return df[['ID']], np.clip(preds, 0.025, 0.975)


print("Getting Men's raw predictions...")
M_ids, M_raw = get_raw_preds(Sub_full, M_stats, MSeeds_clean, M_massey, M_recent,
                              M_tempo, M_sos, M_elo,
                              M_xgb_lean, M_lgb_lean, M_lr_lean,
                              gender_prefix='M', with_massey=True)

print("Getting Women's raw predictions...")
W_ids, W_raw = get_raw_preds(Sub_full, W_stats, WSeeds_clean, None, W_recent,
                              W_tempo, W_sos, W_elo,
                              W_xgb_lean, W_lgb_lean, W_lr_lean,
                              gender_prefix='W', with_massey=False)

# Blend with seed prior
print("\nBlending with seed prior (alpha=0.25)...")
M_ids['Pred'] = blend_with_seed_prior(M_ids, M_raw, MSeeds_clean, alpha=0.25)
W_ids['Pred'] = blend_with_seed_prior(W_ids, W_raw, WSeeds_clean, alpha=0.25)

# Combine
final_v3 = pd.concat([M_ids, W_ids], ignore_index=True)
final_v3  = Sub_full[['ID']].merge(final_v3, on='ID', how='left')
final_v3['Pred'] = final_v3['Pred'].fillna(0.5).clip(0.025, 0.975)

final_v3.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\nShape : {final_v3.shape}")
print(f"Nulls : {final_v3['Pred'].isnull().sum()}")
print(f"Min   : {final_v3['Pred'].min():.4f}")
print(f"Max   : {final_v3['Pred'].max():.4f}")
print(f"Mean  : {final_v3['Pred'].mean():.4f}")
print("Saved to /kaggle/working/submission.csv")

Getting Men's raw predictions...
Getting Women's raw predictions...

Blending with seed prior (alpha=0.25)...

Shape : (519144, 2)
Nulls : 0
Min   : 0.0250
Max   : 0.9750
Mean  : 0.3188
Saved to /kaggle/working/submission.csv


In [20]:
def build_regular_season_matchups(regular_df, stats_df, seeds_df, massey_df,
                                   recent_df, tempo_df, sos_df, elo_dict,
                                   with_massey=True, sample_frac=0.15, seed=42):
    # Sample a fraction of regular season games to avoid memory issues
    # We weight toward recent seasons — they matter more
    sampled = (regular_df.groupby('Season', group_keys=False)
                          .apply(lambda x: x.sample(frac=sample_frac, random_state=seed)))

    rows = []
    for _, row in sampled.iterrows():
        w, l   = row['WTeamID'], row['LTeamID']
        t1, t2 = (w, l) if w < l else (l, w)
        rows.append({'Season': row['Season'], 'Team1': t1, 'Team2': t2,
                     'Label': 1 if w == t1 else 0})
    df = pd.DataFrame(rows)

    # Elo
    elo_rows = [{'Season': s, 'TeamID': t, 'Elo': r}
                for s, ed in elo_dict.items() for t, r in ed.items()]
    elo_df = pd.DataFrame(elo_rows)
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        df = df.merge(elo_df.rename(columns={'TeamID': col, 'Elo': f'{prefix}_Elo'}),
                      on=['Season', col], how='left')
    df['T1_Elo'] = df['T1_Elo'].fillna(1500)
    df['T2_Elo'] = df['T2_Elo'].fillna(1500)
    df['EloDiff'] = df['T1_Elo'] - df['T2_Elo']

    # Season stats
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = stats_df.rename(columns={'TeamID': col})[['Season', col] + STAT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in STAT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')

    # Seeds
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = seeds_df.rename(columns={'TeamID': col})[['Season', col, 'SeedNum']]
        tmp = tmp.rename(columns={'SeedNum': f'{prefix}_SeedNum'})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SeedDiff'] = df['T1_SeedNum'] - df['T2_SeedNum']

    # Massey
    if with_massey and massey_df is not None:
        for prefix, col in [('T1','Team1'),('T2','Team2')]:
            tmp = massey_df.rename(columns={'TeamID': col})[['Season', col, 'MeanRank','MinRank']]
            tmp = tmp.rename(columns={'MeanRank': f'{prefix}_MeanRank',
                                      'MinRank' : f'{prefix}_MinRank'})
            df  = df.merge(tmp, on=['Season', col], how='left')
        df['MeanRankDiff'] = df['T1_MeanRank'] - df['T2_MeanRank']
    else:
        df['T1_MeanRank']=175; df['T2_MeanRank']=175
        df['T1_MinRank'] =175; df['T2_MinRank'] =175
        df['MeanRankDiff']=0

    # Recent form
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = recent_df.rename(columns={'TeamID': col})[['Season', col] + RECENT_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in RECENT_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['RecentWinRateDiff'] = df['T1_RecentWinRate'] - df['T2_RecentWinRate']
    df['RecentPtsDiffDiff'] = df['T1_RecentPtsDiff'] - df['T2_RecentPtsDiff']

    # Tempo & efficiency
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = tempo_df.rename(columns={'TeamID': col})[['Season', col] + TEMPO_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in TEMPO_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['NetEffDiff'] = df['T1_NetEff'] - df['T2_NetEff']
    df['TempoDiff']  = df['T1_Tempo']  - df['T2_Tempo']
    df['OffEffDiff'] = df['T1_OffEff'] - df['T2_OffEff']
    df['DefEffDiff'] = df['T1_DefEff'] - df['T2_DefEff']

    # Strength of schedule
    for prefix, col in [('T1','Team1'),('T2','Team2')]:
        tmp = sos_df.rename(columns={'TeamID': col})[['Season', col] + SOS_COLS]
        tmp = tmp.rename(columns={c: f'{prefix}_{c}' for c in SOS_COLS})
        df  = df.merge(tmp, on=['Season', col], how='left')
    df['SOSDiff']    = df['T1_SOS']    - df['T2_SOS']
    df['SOS_PDDiff'] = df['T1_SOS_PD'] - df['T2_SOS_PD']

    df['WinRateDiff'] = df['T1_WinRate']    - df['T2_WinRate']
    df['PtsDiffDiff'] = df['T1_AvgPtsDiff'] - df['T2_AvgPtsDiff']

    for c in df.columns:
        if 'Rank' in c:   df[c] = df[c].fillna(175)
        elif 'Seed' in c: df[c] = df[c].fillna(16)
    df[FEATURE_COLS_LEAN] = df[FEATURE_COLS_LEAN].fillna(df[FEATURE_COLS_LEAN].median())

    return df


print("Building Men's regular season training data (15% sample)...")
M_reg_train = build_regular_season_matchups(
    MRegular, M_stats, MSeeds_clean, M_massey,
    M_recent, M_tempo, M_sos, M_elo, with_massey=True)

print("Building Women's regular season training data (15% sample)...")
W_reg_train = build_regular_season_matchups(
    WRegular, W_stats, WSeeds_clean, None,
    W_recent, W_tempo, W_sos, W_elo, with_massey=False)

# Combine regular season + tournament
# Weight tournament games 3x — they are what we are predicting
M_combined = pd.concat([M_reg_train,
                         pd.concat([M_train_v2]*3)], ignore_index=True)
W_combined = pd.concat([W_reg_train,
                         pd.concat([W_train_v2]*3)], ignore_index=True)

print(f"\nM_combined: {M_combined.shape}")
print(f"W_combined: {W_combined.shape}")

X_men_comb   = M_combined[FEATURE_COLS_LEAN]
y_men_comb   = M_combined['Label'].values
X_women_comb = W_combined[FEATURE_COLS_LEAN]
y_women_comb = W_combined['Label'].values

print("\n" + "="*45)
print("Men's Combined Model")
print("="*45)
M_xgb_comb, M_lgb_comb, M_lr_comb = train_ensemble(X_men_comb, y_men_comb)

print("\n" + "="*45)
print("Women's Combined Model")
print("="*45)
W_xgb_comb, W_lgb_comb, W_lr_comb = train_ensemble(X_women_comb, y_women_comb)

Building Men's regular season training data (15% sample)...
Building Women's regular season training data (15% sample)...

M_combined: (26168, 73)
W_combined: (17979, 73)

Men's Combined Model
  Fold 1 | Brier: 0.16314 | LogLoss: 0.48800
  Fold 2 | Brier: 0.16823 | LogLoss: 0.50240
  Fold 3 | Brier: 0.16394 | LogLoss: 0.49097
  Fold 4 | Brier: 0.16254 | LogLoss: 0.48699
  Fold 5 | Brier: 0.16773 | LogLoss: 0.50056

  OOF Brier  : 0.16512
  OOF LogLoss: 0.49378

Women's Combined Model
  Fold 1 | Brier: 0.13648 | LogLoss: 0.41809
  Fold 2 | Brier: 0.13946 | LogLoss: 0.42192
  Fold 3 | Brier: 0.13505 | LogLoss: 0.41561
  Fold 4 | Brier: 0.13666 | LogLoss: 0.42224
  Fold 5 | Brier: 0.14365 | LogLoss: 0.43661

  OOF Brier  : 0.13826
  OOF LogLoss: 0.42289


In [21]:
print("Building Men's submission...")
M_ids, M_raw = get_raw_preds(Sub_full, M_stats, MSeeds_clean, M_massey, M_recent,
                              M_tempo, M_sos, M_elo,
                              M_xgb_comb, M_lgb_comb, M_lr_comb,
                              gender_prefix='M', with_massey=True)

print("Building Women's submission...")
W_ids, W_raw = get_raw_preds(Sub_full, W_stats, WSeeds_clean, None, W_recent,
                              W_tempo, W_sos, W_elo,
                              W_xgb_comb, W_lgb_comb, W_lr_comb,
                              gender_prefix='W', with_massey=False)

M_ids['Pred'] = M_raw
W_ids['Pred'] = W_raw

final_v4 = pd.concat([M_ids, W_ids], ignore_index=True)
final_v4  = Sub_full[['ID']].merge(final_v4, on='ID', how='left')
final_v4['Pred'] = final_v4['Pred'].fillna(0.5).clip(0.025, 0.975)

final_v4.to_csv('/kaggle/working/submission.csv', index=False)

print(f"\nShape : {final_v4.shape}")
print(f"Nulls : {final_v4['Pred'].isnull().sum()}")
print(f"Min   : {final_v4['Pred'].min():.4f}")
print(f"Max   : {final_v4['Pred'].max():.4f}")
print(f"Mean  : {final_v4['Pred'].mean():.4f}")
print("Saved to /kaggle/working/submission.csv")

Building Men's submission...
Building Women's submission...

Shape : (519144, 2)
Nulls : 0
Min   : 0.0250
Max   : 0.9750
Mean  : 0.4897
Saved to /kaggle/working/submission.csv


In [22]:
def train_tournament_only(X, y, seasons, n_splits=5, seed=42):
    # Temporal split — validate on most recent seasons
    # This mirrors real world: train on past, predict future
    unique_seasons = sorted(np.unique(seasons))
    print(f"  Seasons in data: {unique_seasons}")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_xgb = np.zeros(len(y))
    oof_lgb = np.zeros(len(y))
    oof_lr  = np.zeros(len(y))

    xgb_models, lgb_models, lr_models = [], [], []

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        Xtr, Xval = X.iloc[tr], X.iloc[val]
        ytr, yval = y[tr], y[val]

        m_xgb = xgb.XGBClassifier(
            n_estimators=1000, learning_rate=0.01, max_depth=3,
            subsample=0.7, colsample_bytree=0.7, min_child_weight=3,
            gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
            eval_metric='logloss', verbosity=0, random_state=seed)
        m_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  verbose=False)
        oof_xgb[val] = m_xgb.predict_proba(Xval)[:,1]
        xgb_models.append(m_xgb)

        m_lgb = lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.01, max_depth=3,
            subsample=0.7, colsample_bytree=0.7, min_child_weight=3,
            reg_alpha=0.1, reg_lambda=1.0,
            random_state=seed, verbose=-1)
        m_lgb.fit(Xtr, ytr, eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(-1)])
        oof_lgb[val] = m_lgb.predict_proba(Xval)[:,1]
        lgb_models.append(m_lgb)

        sc   = StandardScaler()
        m_lr = LogisticRegression(C=0.05, max_iter=1000, random_state=seed)
        m_lr.fit(sc.fit_transform(Xtr), ytr)
        oof_lr[val] = m_lr.predict_proba(sc.transform(Xval))[:,1]
        lr_models.append((sc, m_lr))

        ens = (oof_xgb[val] + oof_lgb[val] + oof_lr[val]) / 3
        print(f"  Fold {fold+1} | Brier: {brier_score_loss(yval, ens):.5f} | LogLoss: {log_loss(yval, ens):.5f}")

    oof_ens = (oof_xgb + oof_lgb + oof_lr) / 3
    print(f"\n  OOF Brier  : {brier_score_loss(y, oof_ens):.5f}")
    print(f"  OOF LogLoss: {log_loss(y, oof_ens):.5f}")

    return xgb_models, lgb_models, lr_models

X_men_t   = M_train_v2[FEATURE_COLS_LEAN]
y_men_t   = M_train_v2['Label'].values
S_men_t   = M_train_v2['Season'].values

X_women_t = W_train_v2[FEATURE_COLS_LEAN]
y_women_t = W_train_v2['Label'].values
S_women_t = W_train_v2['Season'].values

print("="*45)
print("Men's Tuned Tournament Model")
print("="*45)
M_xgb_t, M_lgb_t, M_lr_t = train_tournament_only(X_men_t, y_men_t, S_men_t)

print()
print("="*45)
print("Women's Tuned Tournament Model")
print("="*45)
W_xgb_t, W_lgb_t, W_lr_t = train_tournament_only(X_women_t, y_women_t, S_women_t)

Men's Tuned Tournament Model
  Seasons in data: [np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Fold 1 | Brier: 0.19314 | LogLoss: 0.56258
  Fold 2 | Brier: 0.18118 | LogLoss: 0.53600
  Fold 3 | Brier: 0.18386 | LogLoss: 0.54152
  Fold 4 | Brier: 0.18703 | LogLoss: 0.55164
  Fold 5 | Brier: 0.18723 | LogLoss: 0.55184

  OOF Brier  : 0.18649
  OOF LogLoss: 0.54872

Women's Tuned Tournament Model
  Seaso

In [23]:
# Load historical tourney results to understand what matchups get scored
MTourney_all = pd.read_csv(f'{DATA}/MNCAATourneyCompactResults.csv')
WTourney_all = pd.read_csv(f'{DATA}/WNCAATourneyCompactResults.csv')

# For the submission, matchups between seeded teams are the ones that get scored
# Teams with seeds 1-16 are in the tournament
# For 2026, we don't know seeds yet but we can use recent seed history

# Check: what is the seed range of teams in actual tournament matchups
M_seed_teams_per_season = MSeeds.groupby('Season')['TeamID'].apply(set).to_dict()
W_seed_teams_per_season = WSeeds.groupby('Season')['TeamID'].apply(set).to_dict()

# For submission rows from 2022-2025, tag which ones involve two seeded teams
Sub_full[['Season','Team1','Team2']] = Sub_full['ID'].str.split('_', expand=True).astype(int)

def is_tournament_matchup(row, seed_teams_dict):
    season = row['Season']
    if season not in seed_teams_dict:
        return False
    seeds = seed_teams_dict[season]
    return (row['Team1'] in seeds) and (row['Team2'] in seeds)

print("Tagging tournament-eligible matchups in submission...")
Sub_full['M_tourney_eligible'] = Sub_full.apply(
    lambda r: is_tournament_matchup(r, M_seed_teams_per_season)
    if r['Team1'] < 3000 else False, axis=1)

Sub_full['W_tourney_eligible'] = Sub_full.apply(
    lambda r: is_tournament_matchup(r, W_seed_teams_per_season)
    if r['Team1'] >= 3000 else False, axis=1)

m_elig = Sub_full['M_tourney_eligible'].sum()
w_elig = Sub_full['W_tourney_eligible'].sum()
total_elig = m_elig + w_elig

print(f"Men's tournament-eligible matchups   : {m_elig:,}")
print(f"Women's tournament-eligible matchups : {w_elig:,}")
print(f"Total eligible                       : {total_elig:,}")
print(f"Total submission rows                : {len(Sub_full):,}")
print(f"Eligible as % of total               : {total_elig/len(Sub_full)*100:.2f}%")

Tagging tournament-eligible matchups in submission...
Men's tournament-eligible matchups   : 9,112
Women's tournament-eligible matchups : 9,112
Total eligible                       : 18,224
Total submission rows                : 519,144
Eligible as % of total               : 3.51%


In [24]:
# Recent seasons matter more — weight them higher
def get_season_weights(seasons, decay=0.85):
    max_season = max(seasons)
    weights = np.array([decay ** (max_season - s) for s in seasons])
    return weights

# Build weighted training set — recent seasons get higher weight
M_seasons = M_train_v2['Season'].values
W_seasons = W_train_v2['Season'].values

M_weights = get_season_weights(M_seasons, decay=0.90)
W_weights = get_season_weights(W_seasons, decay=0.90)

print("Men's season weight distribution:")
weight_df = pd.DataFrame({'Season': M_seasons, 'Weight': M_weights})
print(weight_df.groupby('Season')['Weight'].mean().tail(10).round(4).to_string())

def train_weighted(X, y, sample_weights, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_xgb = np.zeros(len(y))
    oof_lgb = np.zeros(len(y))
    oof_lr  = np.zeros(len(y))

    xgb_models, lgb_models, lr_models = [], [], []

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        Xtr, Xval   = X.iloc[tr], X.iloc[val]
        ytr, yval   = y[tr], y[val]
        wtr         = sample_weights[tr]

        m_xgb = xgb.XGBClassifier(
            n_estimators=1000, learning_rate=0.01, max_depth=3,
            subsample=0.7, colsample_bytree=0.7,
            min_child_weight=3, gamma=0.1,
            reg_alpha=0.1, reg_lambda=1.0,
            eval_metric='logloss', verbosity=0, random_state=seed)
        m_xgb.fit(Xtr, ytr, sample_weight=wtr,
                  eval_set=[(Xval, yval)], verbose=False)
        oof_xgb[val] = m_xgb.predict_proba(Xval)[:,1]
        xgb_models.append(m_xgb)

        m_lgb = lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.01, max_depth=3,
            subsample=0.7, colsample_bytree=0.7,
            min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
            random_state=seed, verbose=-1)
        m_lgb.fit(Xtr, ytr, sample_weight=wtr,
                  eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(100, verbose=False),
                             lgb.log_evaluation(-1)])
        oof_lgb[val] = m_lgb.predict_proba(Xval)[:,1]
        lgb_models.append(m_lgb)

        sc   = StandardScaler()
        m_lr = LogisticRegression(C=0.05, max_iter=1000, random_state=seed)
        m_lr.fit(sc.fit_transform(Xtr), ytr, sample_weight=wtr)
        oof_lr[val] = m_lr.predict_proba(sc.transform(Xval))[:,1]
        lr_models.append((sc, m_lr))

        ens = (oof_xgb[val] + oof_lgb[val] + oof_lr[val]) / 3
        print(f"  Fold {fold+1} | Brier: {brier_score_loss(yval, ens):.5f} | LogLoss: {log_loss(yval, ens):.5f}")

    oof_ens = (oof_xgb + oof_lgb + oof_lr) / 3
    print(f"\n  OOF Brier  : {brier_score_loss(y, oof_ens):.5f}")
    print(f"  OOF LogLoss: {log_loss(y, oof_ens):.5f}")

    return xgb_models, lgb_models, lr_models

print("\n" + "="*45)
print("Men's Season-Weighted Model")
print("="*45)
M_xgb_w, M_lgb_w, M_lr_w = train_weighted(
    X_men_t, y_men_t, M_weights)

print("\n" + "="*45)
print("Women's Season-Weighted Model")
print("="*45)
W_xgb_w, W_lgb_w, W_lr_w = train_weighted(
    X_women_t, y_women_t, W_weights)

Men's season weight distribution:
Season
2015    0.3487
2016    0.3874
2017    0.4305
2018    0.4783
2019    0.5314
2021    0.6561
2022    0.7290
2023    0.8100
2024    0.9000
2025    1.0000

Men's Season-Weighted Model
  Fold 1 | Brier: 0.19721 | LogLoss: 0.57496
  Fold 2 | Brier: 0.18735 | LogLoss: 0.55506
  Fold 3 | Brier: 0.18823 | LogLoss: 0.55427
  Fold 4 | Brier: 0.19448 | LogLoss: 0.57308
  Fold 5 | Brier: 0.19504 | LogLoss: 0.57112

  OOF Brier  : 0.19246
  OOF LogLoss: 0.56570

Women's Season-Weighted Model
  Fold 1 | Brier: 0.15379 | LogLoss: 0.47017
  Fold 2 | Brier: 0.15196 | LogLoss: 0.45830
  Fold 3 | Brier: 0.14426 | LogLoss: 0.44133
  Fold 4 | Brier: 0.13796 | LogLoss: 0.42768
  Fold 5 | Brier: 0.15917 | LogLoss: 0.47563

  OOF Brier  : 0.14943
  OOF LogLoss: 0.45463
